# Train on all data and Export the model

## Rerunning the earlier steps

We will be rerunning everything but using our picked model: RandomForest

In [84]:
import pandas as pd
from taxipred.utils.constants import NEW_CSV_PATH

df = pd.read_csv(NEW_CSV_PATH)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       876 non-null    float64
 1   time_of_day            878 non-null    object 
 2   day_of_week            880 non-null    object 
 3   traffic_conditions     875 non-null    object 
 4   weather                880 non-null    object 
 5   trip_duration_minutes  879 non-null    float64
 6   trip_price             925 non-null    float64
dtypes: float64(3), object(4)
memory usage: 50.7+ KB


In [85]:
# X, y - NO train/test split since we're training on ALL data
X = df.drop(columns=["trip_price"])
y = df["trip_price"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nIncluded features: {list(X.columns)}")

Features shape: (925, 6)
Target shape: (925,)

Included features: ['trip_distance_km', 'time_of_day', 'day_of_week', 'traffic_conditions', 'weather', 'trip_duration_minutes']


- Imputation

In [86]:
# Picking numerical and categorical features
numeric_columns = X.select_dtypes(include=["number"]).columns
cat_columns = X.select_dtypes(include=["object"]).columns

numeric_columns, cat_columns

(Index(['trip_distance_km', 'trip_duration_minutes'], dtype='object'),
 Index(['time_of_day', 'day_of_week', 'traffic_conditions', 'weather'], dtype='object'))

- Imputation

In [87]:
# Picking numerical and categorical features
numeric_columns = X.select_dtypes(include=["number"]).columns
cat_columns = X.select_dtypes(include=["object"]).columns

numeric_columns, cat_columns

(Index(['trip_distance_km', 'trip_duration_minutes'], dtype='object'),
 Index(['time_of_day', 'day_of_week', 'traffic_conditions', 'weather'], dtype='object'))

- Imputing null values for the numerical columns

In [88]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression

# Impute numerical columns on ALL data
X_num = X[numeric_columns].copy()

mice_imputer = IterativeImputer(
    estimator=LinearRegression(),
    random_state=42,
    max_iter=20,
    sample_posterior=False
)

X_num_imp = pd.DataFrame(
    mice_imputer.fit_transform(X_num),
    columns=X_num.columns,
    index=X_num.index
)

X_num_imp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       925 non-null    float64
 1   trip_duration_minutes  925 non-null    float64
dtypes: float64(2)
memory usage: 14.6 KB


- Imputing null values for the categorical columns

In [89]:
# Impute categorical columns on ALL data
X_cat = X[cat_columns].copy()

cat_imputer = SimpleImputer(strategy="most_frequent")

X_cat_imp = pd.DataFrame(
    cat_imputer.fit_transform(X_cat),
    columns=cat_columns,
    index=X.index
)

In [90]:
# Combine imputed numerical and categorical columns back together
X_imputed = pd.concat([X_num_imp, X_cat_imp], axis=1)

# Ensure columns are in the same order as original
X_imputed = X_imputed[X.columns]

print("Missing values in X_imputed:", X_imputed.isna().sum().sum())
X_imputed.info()

Missing values in X_imputed: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trip_distance_km       925 non-null    float64
 1   time_of_day            925 non-null    object 
 2   day_of_week            925 non-null    object 
 3   traffic_conditions     925 non-null    object 
 4   weather                925 non-null    object 
 5   trip_duration_minutes  925 non-null    float64
dtypes: float64(2), object(4)
memory usage: 43.5+ KB


- Dummies for Categorical Features

In [91]:
# Create dummy variables for categorical features
X_final = pd.get_dummies(X_imputed, columns=cat_columns, drop_first=True)

print(f"Final features shape: {X_final.shape}")
print(f"Final features: {list(X_final.columns)}")

Final features shape: (925, 10)
Final features: ['trip_distance_km', 'trip_duration_minutes', 'time_of_day_Evening', 'time_of_day_Morning', 'time_of_day_Night', 'day_of_week_Weekend', 'traffic_conditions_Low', 'traffic_conditions_Medium', 'weather_Rain', 'weather_Snow']


## Train!

In [92]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest on ALL data
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_final, y)

print(f"Number of features: {X_final.shape[1]}")
print(f"Number of samples: {X_final.shape[0]}")


Number of features: 10
Number of samples: 925


In [93]:
import joblib 
from taxipred.utils.constants import MODEL_PATH
joblib.dump(rf_model, MODEL_PATH /  "model.joblib")

['/home/apollonspc/taxi-prediction-fullstack-omer-aytug/src/taxipred/models/model.joblib']